In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/databricks_fmcg_data_engineering/Setup/utilities

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
bronze_gross_price_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_gross_price_table = f"{catalog}.{silver_schema}.{data_source}"
gold_products_table = f"{catalog}.{silver_schema}.dim_products"


In [0]:
silver_df = spark.sql(
    f"SELECT * FROM {bronze_gross_price_table}"
)

display(silver_df,10)

In [0]:
silver_df = silver_df.withColumn(
    "date",
    coalesce(
        try_to_date(col("month"),"yyyy-MM-dd"),
        try_to_date(col("month"),"dd-MM-yyyy"),
        try_to_date(col("month"),"yyyy/MM/dd"),
        try_to_date(col("month"),"MM/dd/yyyy"),
        try_to_date(col("month"),"dd/MM/yyyy"),
        )
)

In [0]:
silver_df = silver_df.withColumn( 
                "month",
                month(col("date"))
).withColumn("year", year(col("date") )).withColumn("day", dayofmonth(col("date")))

In [0]:
silver_df = silver_df.withColumn(
    "gross_price",
    trim(col("gross_price"))
)


In [0]:
silver_df = silver_df.select("product_id","date","year","month","day","gross_price","ingest_timestamp","file_name","file_size")
display(silver_df)

In [0]:
silver_df = silver_df.withColumn(
    "gross_price",
    when(
        col("gross_price").rlike("^[0-9]+(\\.[0-9]+)?$"),
        col("gross_price")
    ).when(
        col("gross_price").rlike("^-[0-9]+(\\.[0-9]+)?$"),
        col("gross_price").cast("double") * -1 
    )
    .otherwise(
        lit(None).cast("double")
    )

)

In [0]:
product_silver_table = spark.table("fmcg.silver.products")

wrong_products = silver_df.alias("target").join(
    product_silver_table.select("product_id","product_code").alias("source"),
    on = "product_id",
    how = "left_anti"                                         
)

wrong_products.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .option("delta.enableChangeDataFeed", "true")\
                .saveAsTable("fmcg.silver.gross_price_invalid_products")

In [0]:
product_silver_table = spark.table("fmcg.silver.products")

silver_df = silver_df.alias("target").join(
    product_silver_table.select("product_id","product_code").alias("source"),
    on = "product_id",
    how = "inner"                                         
).select(
 'product_id',
 'product_code',
 'date',
 'year',
 'month',
 'day',
 'gross_price',
 'ingest_timestamp',
 'file_name',
 'file_size',
 )


In [0]:
display(silver_df)

In [0]:
silver_df.write.format("delta")\
    .mode("overwrite")\
        .option("mergeSchema","true")\
            .option("delta.enableChangeDataFeed", "true")\
                .saveAsTable(silver_gross_price_table)
